# test weather API hooks

This notebook testes the various PAI hooks for weather data and aims at studying correlation witzh ground weather data

In [1]:
import pandas as pd
import datetime as dt

In [2]:
from src.api_hooks.weather_sources import _fetch_era5

In [3]:
# data sources
local_weather_src =  "data/raw/land_management/weather_observations.csv"

In [4]:
# import and preprocess local_weather pattern

local_weather = pd.read_csv(local_weather_src)
local_weather["datetime"] = local_weather["timestamp"].apply(lambda x: dt.datetime.fromtimestamp(x))


def preprocess_locasl_weather(local_src: str)->pd.Dataframe:
    local_weather = pd.read_csv(local_weather_src)
    local_weather["datetime"] = local_weather["timestamp"].apply(lambda x: dt.datetime.fromtimestamp(x))
    return local_weather

In [5]:
st_date= local_weather["datetime"].sort_values().iloc[0]
en_date= local_weather["datetime"].sort_values().iloc[-1]
st_date, en_date

(Timestamp('2023-02-19 02:00:00'), Timestamp('2026-03-26 19:14:00'))

## Testing weather sources
4 weather data sources were collected for comparison with the local weather station data: [era5](https://earthdatahub.destine.eu/collections/era5), [imerg](https://gpm.nasa.gov/data/imerg), [prism](https://prism.oregonstate.edu/) and [texmesonet](https://txwaterdatahub.org)


API hooks were developed for all of them.

Here we will compare them with the loca data to see which one is closer to local data

In [29]:
from src.api_hooks import get_precipitation

In [6]:
location_point= {"type": "Point", "coordinates": [-105.0885872, 30.8128463]}
st_date_str = dt.datetime.strftime(st_date, '%Y-%m-%d')
en_date_str = "2023-12-31"

Redoing all api hooks, beacuse Claude was not able to give me somthing that works

In [1]:
from src.api_hooks._utils import Context

CONTEXT = Context("src/api_hooks/settings.toml")
CONTEXT.time_frame=[dt.datetime.strftime(st_date, '%Y-%m-%d'), dt.datetime.strftime(en_date, '%Y-%m-%d')]
CONTEXT.time_frame

NameError: name 'dt' is not defined

## ERA5 donwload

In [2]:
from src.api_hooks._era5 import fetch_era5, process_era5

In [3]:
raw=fetch_era5(CONTEXT)
daily=process_era5(raw, CONTEXT)

# IMERG

In [1]:
from src.api_hooks._gpm_imerg import fetch_imerg, process_imerg
from src.api_hooks._utils import Context
import pathlib as p
context = Context("src/api_hooks/settings.toml")
raw_path = fetch_imerg(context, save_raw=True)
ds = process_imerg(raw_path, context, save_processed=False)
ds.to_netcdf(p.Path("data/raw/weather/processed") / f"imerg_processed_{context.time_frame[0]}-{context.time_frame[1]}.nc4")

/home/matt/miniconda3/envs/wasp/lib/python3.14/site-packages/earthaccess/results.py:343: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()
/home/matt/miniconda3/envs/wasp/lib/python3.14/site-packages/earthaccess/store.py:832: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  total_size = round(sum(granule.size() for granule in granules) / 1024, 2)


QUEUEING TASKS | :   0%|          | 0/973 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/973 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/973 [00:00<?, ?it/s]

## PRISM

In [2]:
from src.api_hooks._prism import fetch_prism, process_prism
raw_path = fetch_prism(context, save_raw=True)
ds = process_prism(raw_path, context, save_processed=False)


In [3]:
import pathlib as p

ds.to_netcdf(p.Path("data/raw/weather/processed") / f"prism_processed_{context.time_frame[0]}-{context.time_frame[1]}.nc")